# Analisis Potensi Banjir Provinsi Bengkulu

Analisis **eksploratif** menggunakan curah hujan (CHIRPS), elevasi (SRTM), dan raster bahaya banjir InaRISK BNPB sebagai pembanding.
Belum ada model prediksi terlatih maupun validasi kuantitatif.

## 1. Setup dan koneksi Earth Engine

In [ ]:
!pip -q install geemap rasterio

import ee
import io, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.io import MemoryFile

try:
    ee.Initialize(project="true-campus-479905-q0")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="true-campus-479905-q0")

print("Google Earth Engine berhasil terhubung!")

## 2. Batas Provinsi Bengkulu (GAUL 2025)

In [ ]:
gaul = ee.FeatureCollection("FAO/GAUL/2025/level1")
bengkulu = (gaul
    .filter(ee.Filter.eq("GAUL0_NAME", "Indonesia"))
    .filter(ee.Filter.eq("GAUL1_NAME", "Bengkulu")))

n = bengkulu.size().getInfo()
print("Jumlah fitur Bengkulu:", n)
assert n == 1, "Batas Bengkulu tidak ditemukan; periksa nama kolom/dataset."

geom = bengkulu.geometry()
bbox = geom.bounds().getInfo()["coordinates"][0]
lons = [p[0] for p in bbox]; lats = [p[1] for p in bbox]
bk_bounds = (min(lons), min(lats), max(lons), max(lats))
print("Bounding box Bengkulu (minx, miny, maxx, maxy):", bk_bounds)

## 3. Curah hujan harian CHIRPS 2024

In [ ]:
hujan = (ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
         .filterDate("2024-01-01", "2025-01-01")
         .select("precipitation"))
print("Jumlah citra harian:", hujan.size().getInfo(), "(diharapkan 366)")

def rata_wilayah(img):
    nilai = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geom,
        scale=5566,
        maxPixels=1e9
    ).get("precipitation")
    return ee.Feature(None, {
        "tanggal": img.date().format("YYYY-MM-dd"),
        "curah_hujan_mm": nilai
    })

fc = ee.FeatureCollection(hujan.map(rata_wilayah))
rows = [f["properties"] for f in fc.getInfo()["features"]]
df_hujan = pd.DataFrame(rows)
df_hujan["tanggal"] = pd.to_datetime(df_hujan["tanggal"])
df_hujan = df_hujan.sort_values("tanggal").reset_index(drop=True)

print("Jumlah baris:", len(df_hujan))
print("Nilai kosong:", df_hujan["curah_hujan_mm"].isna().sum())
df_hujan.head()

## 4. Grafik hujan harian dan bulanan

In [ ]:
nama_bulan = ["Jan","Feb","Mar","Apr","Mei","Jun","Jul","Agu","Sep","Okt","Nov","Des"]

# Grafik harian
plt.figure(figsize=(12, 4))
plt.plot(df_hujan["tanggal"], df_hujan["curah_hujan_mm"], linewidth=0.8)
plt.title("Estimasi curah hujan harian rata-rata wilayah Bengkulu, 2024")
plt.xlabel("Tanggal"); plt.ylabel("mm/hari")
plt.tight_layout(); plt.savefig("grafik_hujan_harian_2024.png", dpi=150); plt.show()

# Ringkasan bulanan (jumlah rata-rata harian wilayah, bukan volume air total)
df_hujan["bulan"] = df_hujan["tanggal"].dt.month
hujan_bulanan = (df_hujan.groupby("bulan")["curah_hujan_mm"]
                 .sum().reindex(range(1, 13), fill_value=0))

plt.figure(figsize=(10, 4))
plt.bar(nama_bulan, hujan_bulanan.values)
plt.title("Estimasi curah hujan bulanan Bengkulu, 2024")
plt.xlabel("Bulan"); plt.ylabel("mm (jumlah rata-rata harian wilayah)")
plt.tight_layout(); plt.savefig("grafik_hujan_bulanan_2024.png", dpi=150); plt.show()

print(hujan_bulanan.round(1).rename(dict(zip(range(1, 13), nama_bulan))))
df_hujan.drop(columns="bulan").to_csv("data_curah_hujan_bengkulu_2024.csv", index=False)

## 5. Elevasi SRTM: statistik

In [ ]:
elevasi = ee.Image("USGS/SRTMGL1_003").select("elevation")
elevasi_bengkulu = elevasi.clip(geom)

reducer = (ee.Reducer.min()
           .combine(ee.Reducer.max(), sharedInputs=True)
           .combine(ee.Reducer.mean(), sharedInputs=True))
stat = elevasi_bengkulu.reduceRegion(
    reducer=reducer, geometry=geom, scale=90, maxPixels=1e10, bestEffort=True
).getInfo()
print("Statistik elevasi Bengkulu (m):")
for k, v in stat.items():
    print(f"  {k}: {v:.1f}")

## 6. Klasifikasi elevasi eksploratif

Batas 50 m dan 200 m adalah **asumsi eksploratif**, bukan standar resmi bahaya banjir.

In [ ]:
kelas = (ee.Image(0)
         .where(elevasi_bengkulu.lt(50), 1)
         .where(elevasi_bengkulu.gte(50).And(elevasi_bengkulu.lt(200)), 2)
         .where(elevasi_bengkulu.gte(200), 3)
         .updateMask(elevasi_bengkulu.mask())
         .rename("kelas"))

luas_img = ee.Image.pixelArea().divide(1e6).addBands(kelas)
hasil = luas_img.reduceRegion(
    reducer=ee.Reducer.sum().group(groupField=1, groupName="kelas"),
    geometry=geom, scale=90, maxPixels=1e10, bestEffort=True
).getInfo()

label = {1: "Rendah (<50 m)", 2: "Sedang (50-<200 m)", 3: "Tinggi (>=200 m)"}
luas = {label[int(g["kelas"])]: g["sum"] for g in hasil["groups"]}
df_luas = pd.DataFrame({"kelas": list(luas), "luas_km2": list(luas.values())})
df_luas["persen"] = df_luas["luas_km2"] / df_luas["luas_km2"].sum() * 100
print(df_luas.round(2))

plt.figure(figsize=(7, 4))
plt.bar(df_luas["kelas"], df_luas["luas_km2"], color=["red", "gold", "green"])
plt.title("Luas kelas elevasi eksploratif, Bengkulu")
plt.ylabel("Luas (km²)")
plt.tight_layout(); plt.savefig("luas_kelas_elevasi.png", dpi=150); plt.show()
df_luas.to_csv("luas_kelas_elevasi_bengkulu.csv", index=False)

## 7. Peta elevasi

In [ ]:
import geemap

peta = geemap.Map(basemap="Esri.WorldImagery", ee_initialize=False)
peta.centerObject(geom, 8)
peta.addLayer(elevasi_bengkulu,
              {"min": 0, "max": 2000, "palette": ["green", "yellow", "brown", "white"]},
              "Elevasi SRTM")
peta.addLayer(kelas,
              {"min": 1, "max": 3, "palette": ["red", "yellow", "green"]},
              "Kelas elevasi", False)
peta.addLayer(bengkulu.style(color="black", fillColor="00000000", width=2), {}, "Batas Bengkulu")
peta

## 8. Memeriksa raster InaRISK (`1_15251.zip`)

Tujuan: memastikan identitas, cakupan, dan NoData raster **sebelum** dipakai membandingkan dengan analisis lain.

In [ ]:
from google.colab import files
uploaded = files.upload()   # pilih 1_15251.zip

In [ ]:
with zipfile.ZipFile("/content/1_15251.zip") as z:
    print("Isi ZIP:", z.namelist())
    tif_name = [n for n in z.namelist() if n.lower().endswith(".tif")][0]
    tif_bytes = z.read(tif_name)

with MemoryFile(tif_bytes) as mf:
    with mf.open() as src:
        print("CRS        :", src.crs)
        print("Ukuran     :", src.width, "x", src.height)
        print("Band       :", src.count)
        print("Resolusi   :", src.res)
        print("Bounds     :", src.bounds)
        print("NoData     :", src.nodata)
        print("Tags       :", src.tags())
        data = src.read(1)
        nodata = src.nodata
        r_bounds = src.bounds

valid = data != nodata if nodata is not None else ~np.isnan(data)
valid &= np.isfinite(data)
nv = int(valid.sum()); nt = data.size
print(f"\nPiksel valid : {nv:,} ({nv/nt*100:.2f}%)")
print(f"Piksel kosong: {nt-nv:,} ({(nt-nv)/nt*100:.2f}%)")
print("Persentil 25/50/75/90/95:",
      np.percentile(data[valid], [25, 50, 75, 90, 95]).round(4))

In [ ]:
# Tumpang tindih cakupan raster dengan bounding box Bengkulu
bx0, by0, bx1, by1 = bk_bounds
rx0, ry0, rx1, ry1 = r_bounds.left, r_bounds.bottom, r_bounds.right, r_bounds.top

ix0, iy0 = max(bx0, rx0), max(by0, ry0)
ix1, iy1 = min(bx1, rx1), min(by1, ry1)
ada_overlap = (ix1 > ix0) and (iy1 > iy0)

luas_bbox_bk = (bx1 - bx0) * (by1 - by0)
luas_overlap = max(0, ix1 - ix0) * max(0, iy1 - iy0)
print("Raster beririsan dengan bbox Bengkulu:", ada_overlap)
print(f"Porsi bbox Bengkulu yang tercakup raster: {luas_overlap/luas_bbox_bk*100:.1f}%")

# Visualisasi: nilai valid saja, NoData transparan
tampil = np.ma.masked_where(~valid, data)
plt.figure(figsize=(6, 6))
plt.imshow(tampil, extent=[rx0, rx1, ry0, ry1], cmap="viridis")
plt.colorbar(label="Nilai piksel (arti belum dikonfirmasi)")
plt.title("Raster 1_15251.tif (NoData transparan)")
plt.xlabel("Bujur"); plt.ylabel("Lintang")
plt.show()

### Catatan interpretasi

- Piksel NoData **bukan** berarti tanpa bahaya banjir.
- Nilai 0-1 belum boleh dikelompokkan menjadi rendah/sedang/tinggi sebelum legenda dan metadata layer dipastikan di layanan BNPB (INDEKS_BAHAYA_BANJIR MapServer).
- Jika cakupan hanya sebagian kecil Bengkulu, hasil perbandingan hanya berlaku untuk area yang tumpang tindih.
- Langkah berikutnya, bila raster sesuai: samakan CRS dan resolusi (`rasterio.warp.reproject`), lalu bandingkan dengan kelas elevasi hanya pada piksel valid.